In [1]:
!pip install timm --no-index --find-links=file:///kaggle/input/libraries/ 


Looking in links: file:///kaggle/input/libraries/
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
ERROR: Could not find a version that satisfies the requirement nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64" (from torch) (from versions: none)
ERROR: No matching distribution found for nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64"


In [2]:
import sys
sys.path.append("../input/saved-weights/pytorch-image-models/pytorch-image-models")


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from torchvision import transforms
from torch import nn
from PIL import Image
from albumentations.pytorch import ToTensorV2
import albumentations.pytorch
import torchvision
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from fastai.vision.all import *
from fastai.data.core import *
import gc
import random

#For deterministic behavious
seed = 999
set_seed(seed, reproducible=True)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
random.seed(seed)
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
def petfinder_rmse(input,target):
    return 100*torch.sqrt(F.mse_loss(F.sigmoid(input.flatten()), target))


In [5]:
base_dir = '/kaggle/input'
model_weights = os.path.join(base_dir, 'saved-weights', 'swin_large_fastai(final).pth')
test_folder = os.path.join(base_dir, 'petfinder-pawpularity-score', 'test')
test_file = os.path.join(base_dir, 'petfinder-pawpularity-score', 'test.csv')


In [6]:
input_shape = (224, 224, 3)
mean, std_dev = [0.5023, 0.4615, 0.4226], [0.2640, 0.2593, 0.2575]
model_name = 'swin_large_patch4_window7_224_in22k'
batch_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
output_categories = 11 # Bins
n_epochs = 15
num_of_hidden = 2
hidden_dimension = [256, 64]
save_name = '/kaggle/working/swin_fastai(final).pth'


In [7]:
test_csv = pd.read_csv(test_file)
test_csv.head()


,Id,Subject Focus,Eyes,Face,Near,Action,Accessory,Group,Collage,Human,Occlusion,Info,Blur
0,ee51b99832f1ba868f646df93d2b6b81,0,1,1,1,0,0,0,0,0,0,0,0
1,caddfb3f8bff9c4b95dbe022018eea21,0,0,1,1,0,1,1,0,0,0,0,0
2,582eeabd4a448a53ebb79995888a4b0b,0,1,1,0,0,0,0,0,0,0,0,0
3,afc1ad7f0c5eea880759d09e77f7deee,0,1,1,1,0,0,0,0,0,0,0,0
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,0,1,1,0,0,0,0,1,0,1,1,0


In [8]:
test_csv['path_img'] = list(map(lambda x: os.path.join(test_folder, x+'.jpg'), test_csv['Id']))
test_csv['Pawpularity'] = [1]*len(test_csv)


In [9]:
class PetsDataset(Dataset):
    
    def __init__(self, df, transform = None, other = False):
        self.transform = transform
        self.df = df
        self.other = other
        self.cat = ['Subject Focus', 'Eyes', 'Face', 'Near', 'Action',
       'Accessory', 'Group', 'Collage', 'Human', 'Occlusion', 'Info', 'Blur']
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df['path_img'].iloc[idx]
        label_1 = self.df['Pawpularity'].iloc[idx]
        img = Image.open(img_path)
        if self.transform:
            album = self.transform(image = np.array(img))
            img = album['image']


        df_data = self.df[self.cat].iloc[idx].values
            
        if self.other:
            return img, label_1, self.df.iloc[idx, 1:-3].to_dict()

        
        return (img, df_data, label_1)


In [10]:
test_transform = A.Compose([A.LongestMaxSize(max_size=448, interpolation=1),
                             A.PadIfNeeded(min_height=input_shape[0], min_width=input_shape[1], border_mode=0, 
                                           value=(0,0,0)),
                             A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
                             A.transforms.ColorJitter(brightness = 0.1, contrast = 0.1, saturation = 0.1, 
                                                      hue = 0.1, p =0.6),
                             A.RandomCrop(height = input_shape[0], width = input_shape[1]),
                             A.HorizontalFlip(p=0.6),
                             A.Normalize(mean, std_dev),
                             ToTensorV2(),
                             ])
test_dataset = PetsDataset(test_csv, test_transform)
testloader = DataLoader(test_dataset, batch_size = batch_size, num_workers = 1, shuffle=False)


AttributeError: 'functools.partial' object has no attribute 'Compose'

In [11]:
dls = DataLoaders.from_dsets(test_dataset)


NameError: name 'test_dataset' is not defined

In [12]:
class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()
        
    def forward(self, x):
        return x

class Network(nn.Module):
    def __init__(self, base, number_of_hidden, hidden, regression_out, output_categories, freeze_layer):
        super(Network, self).__init__()
#         print(number_of_hidden, hidden)
        if (number_of_hidden != len(hidden)):
            raise "Number of Hidden layer and length of hidden dim must be same"
            
        
        hidden_dim = hidden[:]
        hidden_dim.insert(0, base.head.in_features+12)
        
        base.head = Identity()
        self.p = 0.5

        self.regression = self.__fully_connected(number_of_hidden, hidden_dim, regression_out)
        
        self.network = self.__freeze_layer(base, freeze_layer)

        self.__initialise_weights()

    def __initialise_weights(self):
        for m in self.regression:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
              
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
                

    def __freeze_layer(self, base, freeze_layer):
        cnt = 0
        for child in base.children():
            cnt+=1
            if cnt > freeze_layer:
                break

            for param in child.parameters():
                param.requires_grad = False
        return base


    def __fully_connected(self, number_of_hidden, hidden_dim, output_categories):
        layers = []
        for i in range(number_of_hidden):
            layers.append(nn.Linear(hidden_dim[i], hidden_dim[i+1]))
            layers.append(nn.GELU())
            layers.append(nn.BatchNorm1d(hidden_dim[i+1]))
            if i != (number_of_hidden-1):
                layers.append(nn.Dropout(self.p))

        layers.append(nn.Linear(hidden_dim[-1], output_categories))
    
        return nn.Sequential(*layers)
        
    def forward(self,x, tab):
        x1 = self.network(x)
        x = torch.cat([x1, tab], dim=1)
        reg = self.regression(x)
        return reg

network = timm.create_model(model_name, pretrained = False)
model = Network(network, num_of_hidden, hidden_dimension, 1, 11, 0)


/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swin_large_patch4_window7_224_in22k to current swin_large_patch4_window7_224.ms_in22k.
  model = create_fn(


In [13]:
def get_learner(dls, model, loss, metric, save_path):
    dls = dls.to(device)
    model = model.to(device)
    learn = Learner(dls, model, loss_func=loss, metrics=metric, 
                    model_dir = save_path).to_fp16()
    return learn


In [14]:
learn = get_learner(dls, model, BCEWithLogitsLossFlat(), petfinder_rmse, save_name)
learn.load(model_weights)


NameError: name 'dls' is not defined

In [15]:
tta_outputs = []
tta_steps = 4
for i in range(tta_steps):
    m = len(testloader)
    running_loss = 0.0
    model.eval()
    final_outputs = []
    with torch.no_grad():
        for images, tabular, _ in testloader:
            output_list = []
            images, tabular = images.to(device), tabular.to(device)
            reg_output = learn.model(images, tabular)
            reg_output = 100*torch.sigmoid(reg_output)
            output = reg_output.detach().to('cpu').numpy().reshape(-1,).tolist()
            final_outputs.extend(output)
            
    tta_outputs.append(final_outputs)


NameError: name 'testloader' is not defined

In [16]:
tta_outputs_arr = np.mean(np.array(tta_outputs), axis = 0)


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [17]:
tta_outputs_arr


nan

In [18]:
test_csv['Pawpularity'] = tta_outputs_arr


In [19]:
test_csv = test_csv[["Id", "Pawpularity"]]


In [20]:
test_csv.head()


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Id,Pawpularity
0,ee51b99832f1ba868f646df93d2b6b81,NaN
1,caddfb3f8bff9c4b95dbe022018eea21,NaN
2,582eeabd4a448a53ebb79995888a4b0b,NaN
3,afc1ad7f0c5eea880759d09e77f7deee,NaN
4,d5bdf3446e86ce4ec67ce7a00f1cccc2,NaN


In [21]:
test_csv.to_csv("submission.csv", index=False)
